## base_model--> non_instruction_model--> instruction_model--> preference_model

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [2]:
base_model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(base_model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
import zipfile
import os
# Path to your zip file
zip_path = "/content/tinyllama-instruction.zip"

# Extract all files
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

In [4]:
model_path = "/content/checkpoint-3"

In [6]:
!pip uninstall -y torchao
!pip install -U torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 106.1 MB/s eta 0:00:00


In [5]:
import torchao
print(torchao.__version__)

0.17.0


In [6]:
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [10]:
prompt = "Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry."

In [11]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [12]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [13]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry. 6) How does AI affect clinical trials? 7) What are some of the key problems with Artificial Intelligence in health care?
Explain how Artificial Intelligence (AI) will change the future of healthcare. 1) Describe the steps that AI takes to learn from a medical record. 2) Discuss what AI can do for the patient at home, such as making a diagnosis on your own. 3)


## Now lets start with prefrence base tuning or preference based alignment

In [14]:
!pip install -U trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [15]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.2 MB/s eta 0:00:00


In [7]:
from trl import DPOTrainer
from transformers import AutoTokenizer,  AutoModelForCausalLM, TrainingArguments
from peft import PeftModel
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import torch

In [8]:
base_model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [9]:
instruction_checkpoint = "/content/checkpoint-3"

In [10]:
# Load dataset
dataset = load_dataset("csv", data_files="/content/pharma_preference_data.csv")["train"]

Generating train split: 0 examples [00:00, ? examples/s]

In [11]:
tokenizer = AutoTokenizer.from_pretrained(base_model)

In [12]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

get_peft_model() → Create a new LoRA during training

PeftModel.from_pretrained() → Load an already-trained LoRA for inference or further training

In [13]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [ ]:
# pref_model_lora = get_peft_model(instruction_model, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [14]:
base_model

'TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T'

In [15]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [16]:
#STEP B: Load Instruction LoRA + merge
model = PeftModel.from_pretrained(model, instruction_checkpoint)

In [17]:
model = model.merge_and_unload()

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:97: UserWarning: Merge lora module to 8-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [18]:
#STEP C: Attach NEW LoRA for preference
pref_model_lora = get_peft_model(model, lora_config)

| Stage           | What You Should Do                       | Wrong Way (you did)      |
| --------------- | ---------------------------------------- | ------------------------ |
| Non-Instruction | Base + LoRA                              | ✔ correct                |
| Instruction     | Base + **merge(stage1 LoRA)** + NEW LoRA | ❌ “LoRA on LoRA”         |
| Preference      | Base + **merge(stage2 LoRA)** + NEW LoRA | ❌ “LoRA on LoRA on LoRA” |


In [19]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [20]:
from trl import DPOTrainer, DPOConfig

In [21]:
dpo_args = DPOConfig(
    output_dir="./tinyllama-preference-alignment",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    beta=0.1,
    report_to=[],
    logging_dir=None, # disable logging to wandb or tensorboard
    loss_type="sigmoid",  # or "hinge", depending on experiment
    remove_unused_columns=False
)


In [22]:
trainer = DPOTrainer(
    model=pref_model_lora,
    ref_model=None,
    args=dpo_args,
    train_dataset=dataset,
    processing_class=tokenizer,   # instead of tokenizer argument
    # you can pass data_collator if needed,
    # optionally eval_dataset etc.
)

Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

In [23]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss


TrainOutput(global_step=1, training_loss=0.6931471824645996, metrics={'train_runtime': 5.5988, 'train_samples_per_second': 0.893, 'train_steps_per_second': 0.179, 'total_flos': 4366854955008.0, 'train_loss': 0.6931471824645996, 'entropy': 2.0842321634292604, 'num_tokens': 566.0, 'logits/chosen': -3.5440226682366416, 'logits/rejected': -3.6391777078549863, 'mean_token_accuracy': 0.5961200177669526, 'rewards/chosen': 0.0, 'rewards/rejected': 0.0, 'rewards/accuracies': 0.0, 'rewards/margins': 0.0, 'logps/chosen': -93.67644958496093, 'logps/rejected': -62.53306274414062, 'epoch': 1.0})

In [24]:
question = "Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment."

### Testing with Instruction-Fine-Tuned Model

In [32]:
model_path = "/content/checkpoint-3"
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [35]:
inputs = tokenizer(question, return_tensors="pt").to("cuda")

In [36]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [37]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment.
Crohn's disease (CD) is an inflammatory bowel disease that can be severe. Recent research has shown that metformin may reduce the risk of flare-ups in people with CD.
What Is Crohn's Disease?
Crohn's disease (CD) is a type of inflammatory bowel disease that is characterized by inflammation in the intestinal tract, which can lead to abdom


### Testing with DPO (Preference-Aligned) Model

In [25]:
model_path = "/content/tinyllama-preference-alignment/checkpoint-1"

In [26]:
preference_aligned_model = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.float16)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [27]:
import os
from google.colab import userdata
from huggingface_hub import login

# Retrieve token from Colab Secrets and log in
hf_token = userdata.get('HF_TOKEN_WRITE')
if hf_token:
    login(token=hf_token)
    print("Successfully authenticated with Hugging Face!")
else:
    print("Error: HF_TOKEN secret not found in Colab.")

Successfully authenticated with Hugging Face!


In [28]:
repo_id= "shashrao54/DPO-preference-aligned-finetuning-model"

In [29]:
preference_aligned_model.push_to_hub(
    repo_id,
    private=False
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  29%|##9       |  667kB / 2.26MB            

CommitInfo(commit_url='https://huggingface.co/shashrao54/DPO-preference-aligned-finetuning-model/commit/f692060c2b683443f33e158831478c0f8fd07b30', commit_message='Upload LlamaForCausalLM', commit_description='', oid='f692060c2b683443f33e158831478c0f8fd07b30', pr_url=None, repo_url=RepoUrl('https://huggingface.co/shashrao54/DPO-preference-aligned-finetuning-model', endpoint='https://huggingface.co', repo_type='model', repo_id='shashrao54/DPO-preference-aligned-finetuning-model'), pr_revision=None, pr_num=None)

In [30]:
tokenizer.push_to_hub(
    repo_id,
    private=False
)

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/shashrao54/DPO-preference-aligned-finetuning-model/commit/a4bcf8122dbe5e48b19d7b7002e3ada1ab2ad100', commit_message='Upload tokenizer', commit_description='', oid='a4bcf8122dbe5e48b19d7b7002e3ada1ab2ad100', pr_url=None, repo_url=RepoUrl('https://huggingface.co/shashrao54/DPO-preference-aligned-finetuning-model', endpoint='https://huggingface.co', repo_type='model', repo_id='shashrao54/DPO-preference-aligned-finetuning-model'), pr_revision=None, pr_num=None)

In [31]:
preference_aligned_model.to("cuda")

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2048, out_features=8, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=8, out_features=2048, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=256, bia

In [32]:
inputs = tokenizer(question, return_tensors="pt").to("cuda")

In [35]:
outputs = preference_aligned_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [36]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment.
Metformin, a common diabetes drug used to treat type 2 diabetes, has long been known as a "fat burner," but new research suggests that it may also help prevent cancer.
In a review of more than 100 studies published this week in the journal Diabetologia, researchers from Britain's National Health Service (NHS) and Oxford University said metformin may have properties that can be exploited to prevent cancer by:
